# ACDADA — Notebook 04: Attack Classification Agent

**Agent 2: Multi-Class Attack Classification**

This notebook implements:
1. XGBoost multi-class classifier with Optuna hyperparameter tuning
2. Deep Neural Network (DNN) multi-class classifier
3. Ensemble (soft voting) of XGBoost + DNN
4. Per-class evaluation (DDoS, DoS, BruteForce, Botnet, PortScan, Injection, Other)
5. Confusion matrix analysis, per-class PR curves
6. Model export for deployment

In [ ]:
# ============================================================
# DATASET LINKS (processed data from Notebook 01)
# ============================================================
#
# Original raw datasets:
# 1. CIC-IDS-2017: https://www.kaggle.com/datasets/chethuhn/network-intrusion-dataset
# 2. UNSW-NB15:    https://www.kaggle.com/datasets/mrwellsdavid/unsw-nb15
# 3. Bot-IoT:      https://www.kaggle.com/datasets/vigneshvenkateswaran/bot-iot-dataset
# 4. BETH:         https://www.kaggle.com/datasets/katehighnam/beth-dataset
#
# Processed data at: ../data/processed/<dataset>/
# Run Notebook 01 first to generate processed splits.
# ============================================================

## 1. Imports & Configuration

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler

import xgboost as xgb
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    f1_score, precision_recall_fscore_support, roc_auc_score
)
from sklearn.model_selection import StratifiedKFold

import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

import joblib
import json
import gc

warnings.filterwarnings('ignore')

# Paths
PROCESSED_DIR = Path('../data/processed')
MODELS_DIR = Path('../models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | Device: {DEVICE}')

RANDOM_STATE = 42
BATCH_SIZE = 512
DNN_EPOCHS = 100
DNN_LR = 1e-3
DNN_PATIENCE = 12
OPTUNA_TRIALS = 50

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

## 2. Load Processed Data

In [ ]:
def load_splits(dataset_name: str, processed_dir: Path) -> dict:
    d = processed_dir / dataset_name
    if not d.exists():
        return None
    data = {}
    for key in ['X_train', 'X_val', 'X_test',
                'y_train_binary', 'y_val_binary', 'y_test_binary',
                'y_train_multi', 'y_val_multi', 'y_test_multi']:
        fp = d / f'{key}.npy'
        if fp.exists():
            data[key] = np.load(fp)
    le_path = d / 'multi_label_encoder.joblib'
    if le_path.exists():
        data['label_encoder'] = joblib.load(le_path)
    return data

DATASET_NAME = 'cicids2017'
for name in ['unified', 'cicids2017', 'unsw_nb15', 'bot_iot']:
    if (PROCESSED_DIR / name).exists():
        DATASET_NAME = name
        break

print(f'Loading: {DATASET_NAME}')
data = load_splits(DATASET_NAME, PROCESSED_DIR)
if data is None:
    raise FileNotFoundError('Run Notebook 01 first.')

X_train = data['X_train']
X_val   = data['X_val']
X_test  = data['X_test']

# Use multi-class labels if available, else binary
if 'y_train_multi' in data:
    y_train = data['y_train_multi']
    y_val   = data['y_val_multi']
    y_test  = data['y_test_multi']
    label_encoder = data.get('label_encoder', None)
    if label_encoder is not None:
        CLASS_NAMES = list(label_encoder.classes_)
    else:
        CLASS_NAMES = [str(c) for c in sorted(np.unique(y_train))]
else:
    y_train = data['y_train_binary']
    y_val   = data['y_val_binary']
    y_test  = data['y_test_binary']
    CLASS_NAMES = ['Benign', 'Attack']
    label_encoder = None

N_FEATURES = X_train.shape[1]
N_CLASSES = len(np.unique(y_train))

print(f'Features: {N_FEATURES} | Classes: {N_CLASSES}')
print(f'Class names: {CLASS_NAMES}')
print(f'Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}')
print(f'Train class dist: {np.bincount(y_train)}')

---
## 3. XGBoost with Optuna Hyperparameter Tuning

In [ ]:
def xgb_objective(trial, X_tr, y_tr, X_v, y_v, n_classes):
    """
    Optuna objective function for XGBoost hyperparameter optimization.
    Maximizes weighted F1 on validation set.
    """
    params = {
        'objective': 'multi:softprob' if n_classes > 2 else 'binary:logistic',
        'num_class': n_classes if n_classes > 2 else None,
        'eval_metric': 'mlogloss' if n_classes > 2 else 'logloss',
        'tree_method': 'hist',
        'device': 'cuda' if torch.cuda.is_available() else 'cpu',
        'random_state': RANDOM_STATE,
        'verbosity': 0,
        
        # Tuned parameters
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000, step=50),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'gamma': trial.suggest_float('gamma', 1e-8, 5.0, log=True),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 0.5, 3.0),
    }
    
    # Remove None values
    params = {k: v for k, v in params.items() if v is not None}
    
    model = xgb.XGBClassifier(**params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_v, y_v)],
        verbose=False,
    )
    
    preds = model.predict(X_v)
    f1 = f1_score(y_v, preds, average='weighted')
    return f1


print('Running Optuna hyperparameter search for XGBoost...')
print(f'  Trials: {OPTUNA_TRIALS}')

study = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=RANDOM_STATE),
    study_name='xgb_attack_classification'
)

study.optimize(
    lambda trial: xgb_objective(trial, X_train, y_train, X_val, y_val, N_CLASSES),
    n_trials=OPTUNA_TRIALS,
    show_progress_bar=True,
)

print(f'\nBest trial:')
print(f'  F1 Score: {study.best_value:.4f}')
print(f'  Params: {study.best_params}')

In [ ]:
# Train final XGBoost with best params
best_params = study.best_params.copy()
best_params.update({
    'objective': 'multi:softprob' if N_CLASSES > 2 else 'binary:logistic',
    'eval_metric': 'mlogloss' if N_CLASSES > 2 else 'logloss',
    'tree_method': 'hist',
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'random_state': RANDOM_STATE,
    'verbosity': 1,
})
if N_CLASSES > 2:
    best_params['num_class'] = N_CLASSES

print('Training final XGBoost model with best hyperparameters...')
xgb_model = xgb.XGBClassifier(**best_params)
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    verbose=50,
)

# Predictions
xgb_preds_val = xgb_model.predict(X_val)
xgb_probs_val = xgb_model.predict_proba(X_val)
xgb_preds_test = xgb_model.predict(X_test)
xgb_probs_test = xgb_model.predict_proba(X_test)

print(f'\nXGBoost Validation Accuracy: {accuracy_score(y_val, xgb_preds_val):.4f}')
print(f'XGBoost Validation F1 (weighted): {f1_score(y_val, xgb_preds_val, average="weighted"):.4f}')

In [ ]:
# Feature importance
importance = xgb_model.feature_importances_
feat_names = [f'feature_{i}' for i in range(N_FEATURES)]

# Try to load actual feature names
fn_path = PROCESSED_DIR / DATASET_NAME / 'feature_names.npy'
if fn_path.exists():
    feat_names = np.load(fn_path, allow_pickle=True).tolist()

fi_df = pd.DataFrame({'feature': feat_names[:len(importance)], 'importance': importance})
fi_df = fi_df.sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(12, 8))
top20 = fi_df.head(20)
ax.barh(top20['feature'], top20['importance'], color='teal')
ax.set_xlabel('Feature Importance (Gain)')
ax.set_title('XGBoost — Top 20 Feature Importances')
ax.invert_yaxis()
plt.tight_layout(); plt.show()

---
## 4. Deep Neural Network Classifier

In [ ]:
class DNNAttackClassifier(nn.Module):
    """
    Deep Neural Network for multi-class attack classification.
    
    Architecture:
    - 4 FC blocks with BatchNorm, GELU, Dropout
    - Residual connections between blocks of same size
    - Output: N_CLASSES logits
    """
    
    def __init__(self, n_features: int, n_classes: int, dropout: float = 0.3):
        super().__init__()
        self.n_features = n_features
        self.n_classes = n_classes
        
        self.input_proj = nn.Sequential(
            nn.Linear(n_features, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        
        self.block1 = nn.Sequential(
            nn.Linear(256, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        
        self.block2 = nn.Sequential(
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        
        self.block3 = nn.Sequential(
            nn.Linear(128, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
        )
        
        self.head = nn.Sequential(
            nn.Linear(128, 64),
            nn.GELU(),
            nn.Linear(64, n_classes),
        )
        
        self._init_weights()
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        x = self.input_proj(x)        # (batch, 256)
        x = x + self.block1(x)         # residual (batch, 256)
        x = self.block2(x)             # (batch, 128)
        x = x + self.block3(x)         # residual (batch, 128)
        x = self.head(x)               # (batch, n_classes)
        return x

# Test
dnn_model = DNNAttackClassifier(N_FEATURES, N_CLASSES).to(DEVICE)
dummy = torch.randn(4, N_FEATURES).to(DEVICE)
out = dnn_model(dummy)
print(f'DNN output: {out.shape}')
print(f'Parameters: {sum(p.numel() for p in dnn_model.parameters()):,}')

In [ ]:
# Data loaders with class weighting
class_counts = np.bincount(y_train)
class_weight_arr = 1.0 / np.maximum(class_counts, 1)
sample_weights = class_weight_arr[y_train]
sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(sample_weights),
    num_samples=len(y_train), replacement=True
)
class_weight_tensor = torch.FloatTensor(
    [len(y_train) / (N_CLASSES * max(c, 1)) for c in class_counts]
).to(DEVICE)

train_loader = DataLoader(
    TensorDataset(torch.FloatTensor(X_train), torch.LongTensor(y_train)),
    batch_size=BATCH_SIZE, sampler=sampler, num_workers=0, pin_memory=True
)
val_loader = DataLoader(
    TensorDataset(torch.FloatTensor(X_val), torch.LongTensor(y_val)),
    batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=0, pin_memory=True
)
test_loader = DataLoader(
    TensorDataset(torch.FloatTensor(X_test), torch.LongTensor(y_test)),
    batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=0, pin_memory=True
)

print(f'Class weights: {class_weight_tensor.cpu().numpy().round(3)}')
print(f'Train batches: {len(train_loader)}')

In [ ]:
# Training
dnn_model = DNNAttackClassifier(N_FEATURES, N_CLASSES, dropout=0.3).to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weight_tensor)
optimizer = optim.AdamW(dnn_model.parameters(), lr=DNN_LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-6)

history = {'train_loss': [], 'val_loss': [], 'val_f1': [], 'val_acc': [], 'lr': []}
best_val_f1 = 0
best_state = None
patience_counter = 0

print(f'Training DNN ({DNN_EPOCHS} epochs, patience={DNN_PATIENCE})...')
print('-' * 80)

for epoch in range(1, DNN_EPOCHS + 1):
    # --- Train ---
    dnn_model.train()
    train_loss = 0
    n_train = 0
    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
        optimizer.zero_grad()
        out = dnn_model(X_b)
        loss = criterion(out, y_b)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(dnn_model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item() * X_b.size(0)
        n_train += X_b.size(0)
    train_loss /= n_train
    
    # --- Validate ---
    dnn_model.eval()
    val_loss = 0; n_val = 0
    all_preds = []; all_labels = []
    with torch.no_grad():
        for X_b, y_b in val_loader:
            X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
            out = dnn_model(X_b)
            val_loss += criterion(out, y_b).item() * X_b.size(0)
            n_val += X_b.size(0)
            all_preds.extend(out.argmax(dim=1).cpu().numpy())
            all_labels.extend(y_b.cpu().numpy())
    val_loss /= n_val
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    val_f1 = f1_score(all_labels, all_preds, average='weighted')
    val_acc = accuracy_score(all_labels, all_preds)
    
    scheduler.step()
    lr = optimizer.param_groups[0]['lr']
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_f1'].append(val_f1)
    history['val_acc'].append(val_acc)
    history['lr'].append(lr)
    
    if epoch % 5 == 0 or epoch == 1:
        print(f'  Epoch {epoch:3d}/{DNN_EPOCHS} | TrLoss: {train_loss:.4f} | '
              f'VlLoss: {val_loss:.4f} Acc: {val_acc:.4f} F1: {val_f1:.4f} | LR: {lr:.2e}')
    
    if val_f1 > best_val_f1 + 1e-4:
        best_val_f1 = val_f1
        best_state = {k: v.cpu().clone() for k, v in dnn_model.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= DNN_PATIENCE:
            print(f'  Early stopping at epoch {epoch}')
            break

if best_state:
    dnn_model.load_state_dict(best_state)
print(f'Best val F1: {best_val_f1:.4f}')

In [ ]:
# Training curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('DNN Attack Classifier — Training Curves', fontsize=14, fontweight='bold')
epochs_r = range(1, len(history['train_loss']) + 1)

axes[0].plot(epochs_r, history['train_loss'], 'b--', label='Train')
axes[0].plot(epochs_r, history['val_loss'], 'r-', label='Val')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_r, history['val_acc'], 'g-', label='Accuracy')
axes[1].plot(epochs_r, history['val_f1'], 'orange', label='F1')
axes[1].set_title('Val Metrics'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].plot(epochs_r, history['lr'], 'purple')
axes[2].set_title('Learning Rate'); axes[2].set_yscale('log'); axes[2].grid(True, alpha=0.3)

for ax in axes:
    ax.set_xlabel('Epoch')
plt.tight_layout(); plt.show()

In [ ]:
# DNN predictions on test
dnn_model.eval()
dnn_preds_test = []
dnn_probs_test = []

with torch.no_grad():
    for X_b, _ in test_loader:
        X_b = X_b.to(DEVICE)
        out = dnn_model(X_b)
        probs = torch.softmax(out, dim=1)
        dnn_preds_test.extend(out.argmax(1).cpu().numpy())
        dnn_probs_test.extend(probs.cpu().numpy())

dnn_preds_test = np.array(dnn_preds_test)
dnn_probs_test = np.array(dnn_probs_test)

print(f'DNN Test Accuracy: {accuracy_score(y_test, dnn_preds_test):.4f}')
print(f'DNN Test F1:       {f1_score(y_test, dnn_preds_test, average="weighted"):.4f}')

---
## 5. Ensemble: XGBoost + DNN (Soft Voting)

In [ ]:
class SoftVotingEnsemble:
    """
    Soft voting ensemble combining XGBoost and DNN probabilities.
    Weights are calibrated on validation set.
    """
    
    def __init__(self):
        self.weights = None
    
    def calibrate(self, probs_dict: dict, y_true):
        """Find optimal weights by grid search on validation set."""
        best_f1 = 0
        best_w = None
        methods = list(probs_dict.keys())
        
        for w1 in np.arange(0.1, 1.0, 0.05):
            w2 = 1.0 - w1
            weights = {methods[0]: w1, methods[1]: w2}
            
            avg_probs = sum(weights[m] * probs_dict[m] for m in methods)
            preds = avg_probs.argmax(axis=1)
            f1 = f1_score(y_true, preds, average='weighted')
            
            if f1 > best_f1:
                best_f1 = f1
                best_w = weights.copy()
        
        self.weights = best_w
        print(f'  Optimal weights: {self.weights}')
        print(f'  Calibration F1: {best_f1:.4f}')
        return best_f1
    
    def predict_proba(self, probs_dict: dict):
        methods = list(probs_dict.keys())
        return sum(self.weights[m] * probs_dict[m] for m in methods)
    
    def predict(self, probs_dict: dict):
        return self.predict_proba(probs_dict).argmax(axis=1)

# DNN val probabilities
dnn_model.eval()
dnn_probs_val = []
with torch.no_grad():
    for X_b, _ in val_loader:
        probs = torch.softmax(dnn_model(X_b.to(DEVICE)), dim=1)
        dnn_probs_val.extend(probs.cpu().numpy())
dnn_probs_val = np.array(dnn_probs_val)

# Calibrate ensemble
ensemble = SoftVotingEnsemble()
ensemble.calibrate(
    {'xgb': xgb_probs_val, 'dnn': dnn_probs_val},
    y_val
)

# Ensemble test predictions
ens_probs_test = ensemble.predict_proba({'xgb': xgb_probs_test, 'dnn': dnn_probs_test})
ens_preds_test = ens_probs_test.argmax(axis=1)

print(f'\nEnsemble Test Accuracy: {accuracy_score(y_test, ens_preds_test):.4f}')
print(f'Ensemble Test F1:       {f1_score(y_test, ens_preds_test, average="weighted"):.4f}')

---
## 6. Comprehensive Test Evaluation

In [ ]:
def full_evaluation(y_true, y_pred, y_probs, class_names, model_name):
    """Full multi-class evaluation with visualizations."""
    print(f'\n{"="*60}')
    print(f'  {model_name} — Test Results')
    print(f'{"="*60}')
    
    acc = accuracy_score(y_true, y_pred)
    f1_w = f1_score(y_true, y_pred, average='weighted')
    f1_macro = f1_score(y_true, y_pred, average='macro')
    
    print(f'  Accuracy:       {acc:.4f}')
    print(f'  F1 (weighted):  {f1_w:.4f}')
    print(f'  F1 (macro):     {f1_macro:.4f}')
    
    # Per-class AUC
    try:
        auc_ovr = roc_auc_score(y_true, y_probs, multi_class='ovr', average='weighted')
        print(f'  AUC-ROC (OVR):  {auc_ovr:.4f}')
    except Exception:
        auc_ovr = 0.0
    
    # Available class names for this data
    present_classes = sorted(np.unique(np.concatenate([np.unique(y_true), np.unique(y_pred)])))
    target_names = [class_names[i] if i < len(class_names) else str(i) for i in present_classes]
    
    print(f'\nClassification Report:')
    print(classification_report(y_true, y_pred, target_names=target_names, digits=4))
    
    return {'model': model_name, 'accuracy': acc, 'f1_weighted': f1_w,
            'f1_macro': f1_macro, 'auc': auc_ovr}

xgb_res  = full_evaluation(y_test, xgb_preds_test, xgb_probs_test, CLASS_NAMES, 'XGBoost')
dnn_res  = full_evaluation(y_test, dnn_preds_test, dnn_probs_test, CLASS_NAMES, 'DNN')
ens_res  = full_evaluation(y_test, ens_preds_test, ens_probs_test, CLASS_NAMES, 'Ensemble (XGB+DNN)')

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(22, 6))
fig.suptitle('Attack Classification — Confusion Matrices', fontsize=14, fontweight='bold')

present_classes = sorted(np.unique(y_test))
tick_labels = [CLASS_NAMES[i] if i < len(CLASS_NAMES) else str(i) for i in present_classes]

for ax, preds, name in [
    (axes[0], xgb_preds_test, 'XGBoost'),
    (axes[1], dnn_preds_test, 'DNN'),
    (axes[2], ens_preds_test, 'Ensemble'),
]:
    cm = confusion_matrix(y_test, preds, labels=present_classes)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', ax=ax,
                xticklabels=tick_labels, yticklabels=tick_labels)
    ax.set_title(name); ax.set_ylabel('True'); ax.set_xlabel('Predicted')

plt.tight_layout(); plt.show()

In [ ]:
# Comparison summary
comp_df = pd.DataFrame([xgb_res, dnn_res, ens_res])
comp_df = comp_df.sort_values('f1_weighted', ascending=False)

print('\n' + '='*60)
print('  ATTACK CLASSIFICATION — MODEL COMPARISON')
print('='*60)
print(comp_df.to_string(index=False))

BEST_MODEL_NAME = comp_df.iloc[0]['model']
print(f'\nBest model: {BEST_MODEL_NAME}')

---
## 7. Save Models & Artifacts

In [ ]:
classification_dir = MODELS_DIR / 'attack_classification'
classification_dir.mkdir(parents=True, exist_ok=True)

# Save XGBoost
joblib.dump({
    'model': xgb_model,
    'best_params': best_params,
    'n_features': N_FEATURES,
    'n_classes': N_CLASSES,
    'class_names': CLASS_NAMES,
    'dataset': DATASET_NAME,
    'timestamp': datetime.now().isoformat(),
}, classification_dir / 'xgboost_classifier.joblib')
print('Saved XGBoost')

# Save DNN
torch.save({
    'model_state_dict': dnn_model.state_dict(),
    'model_class': 'DNNAttackClassifier',
    'n_features': N_FEATURES,
    'n_classes': N_CLASSES,
    'class_names': CLASS_NAMES,
    'history': history,
    'dataset': DATASET_NAME,
    'timestamp': datetime.now().isoformat(),
}, classification_dir / 'dnn_classifier.pth')
print('Saved DNN')

# Save ensemble config
joblib.dump({
    'weights': ensemble.weights,
    'methods': ['xgb', 'dnn'],
    'n_classes': N_CLASSES,
    'class_names': CLASS_NAMES,
    'dataset': DATASET_NAME,
    'timestamp': datetime.now().isoformat(),
}, classification_dir / 'ensemble_config.joblib')
print('Saved ensemble config')

# Save label encoder
if label_encoder is not None:
    joblib.dump(label_encoder, classification_dir / 'label_encoder.joblib')
    print('Saved label encoder')

# Save Optuna study
joblib.dump(study, classification_dir / 'optuna_study.joblib')

# Save comparison
comp_df.to_csv(classification_dir / 'classification_comparison.csv', index=False)

print(f'\nAll models saved to {classification_dir}')

In [ ]:
# Inference helper
def load_attack_classifier(models_dir: str, device: str = 'cpu'):
    """
    Load attack classification models for inference.
    Returns xgb_model, dnn_model, ensemble_config, class_names
    """
    p = Path(models_dir)
    
    xgb_data = joblib.load(p / 'xgboost_classifier.joblib')
    dnn_ckpt = torch.load(p / 'dnn_classifier.pth', map_location=device, weights_only=False)
    
    dnn = DNNAttackClassifier(dnn_ckpt['n_features'], dnn_ckpt['n_classes'])
    dnn.load_state_dict(dnn_ckpt['model_state_dict'])
    dnn.to(device).eval()
    
    ens = joblib.load(p / 'ensemble_config.joblib')
    
    return xgb_data['model'], dnn, ens, dnn_ckpt['class_names']

print('\n✓ Notebook 04 complete. Ready for Notebook 05 (Deception Environment).')

In [ ]:
gc.collect()
if DEVICE.type == 'cuda':
    torch.cuda.empty_cache()
print('Memory freed.')